In [ ]:
# Step 1 — Text cleaning (procedural)
# ----------------------------------
# Usage: set `csv_path` to your CSV file, run the cell. It will produce `data_cleaned.csv`.

import re
import pandas as pd

# emoji regex (covers common ranges)
_emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "]+",
    flags=re.UNICODE
)

def clean_text_proc(text: str) -> str:
    """Clean a single text string: remove URLs, markdown links, hashtags (keep words), emojis, and extra whitespace."""
    if not isinstance(text, str):
        return ""
    # remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # convert markdown links [label](url) -> label
    text = re.sub(r'\[(.*?)\]\(.*?\)', lambda m: m.group(1), text)
    # remove leading/trailing hashes while keeping the word
    text = re.sub(r'#(\w+)', lambda m: m.group(1), text)
    # strip emojis
    text = _emoji_pattern.sub('', text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Procedural flow
csv_path = 'data/channel_data.csv' 
cleaned_path = 'data/channel_data_cleaned.csv'

print('Loading', csv_path)
df = pd.read_csv(csv_path, parse_dates=['date'])
print('Rows loaded:', len(df))

print('Cleaning text...')
df['text_clean'] = df['text'].apply(clean_text_proc)

print('Sample cleaned texts:')
print(df[['id', 'text_clean']].head().to_string())

print('Saving cleaned CSV to', cleaned_path)
df.to_csv(cleaned_path, index=False)
print('Done.')


In [1]:
!pip install faiss-cpu

In [2]:
ULTRA_STRICT_PROMPT = """You are an Information Extraction Engine operating in STRICT MODE.
You MUST output ONLY valid LightRAG extraction lines.

CRITICAL RULES (MUST FOLLOW):
1. Output ONLY the allowed formats below.
2. NO markdown, NO punctuation outside fields, NO bullets, NO commentary.
3. NO introductory text, NO explanations, NO blank lines.
4. DO NOT output anything before the first entity or relation line.
5. DO NOT generate any lines that do not strictly match the required formats.

VALID OUTPUT FORMATS (ONLY these):

For ENTITIES (exactly 4 fields):
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>

For RELATIONS (exactly 5 fields):
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

FIELD RULES:
- You MUST NOT use "#", "|", "#|" inside fields.
- Do NOT leave any field empty. Use "N/A" if no extra information.
- <entity_type> MUST be one of: person, organization, location, concept, object, event, role.
- <relation_type> MUST be a simple verb‑like string (e.g., "uses", "creates", "belongs_to", "mentions").
- <entity_name> is the name of the entity (e.g., "Apple Inc", "Tim Cook")
- <extra_info> can be additional context or description

EXAMPLES:
entity#|#|Apple Inc#|#|organization#|#|technology company
entity#|#|Tim Cook#|#|person#|#|CEO of Apple
relation#|#|is_ceo_of#|#|Tim Cook#|#|Apple Inc#|#|current CEO

TERMINATION:
After ALL extraction lines, output EXACTLY:
<|COMPLETE|>

If NOTHING is extractable, output ONLY:
<|COMPLETE|>

Begin extraction now.
"""

In [ ]:
import os

# -----------------------------
# Параметры GPTunneL
# -----------------------------
GPTUNNEL_API_KEY = os.getenv("GPTUNNEL_API_KEY", "Ваш ключ")
GPTUNNEL_URL = "https://gptunnel.ru/v1/chat/completions"
# Выбери модель, доступную через GPTunneL
# Можно посмотреть список через GET /v1/models :contentReference[oaicite:1]{index=1}
GPTUNNEL_MODEL = "gpt-3.5-turbo"  

In [4]:
import os
import asyncio
import logging
import re
import pandas as pd
from typing import Optional, List, Dict

from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status
from lightrag import QueryParam  # <- важно

# Параметры GPTunnel
#GPTUNNEL_API_KEY = os.getenv("GPTUNNEL_API_KEY", "<YOUR_KEY>")
#GPTUNNEL_URL = "https://gptunnel.ru/v1/chat/completions"
#GPTUNNEL_MODEL = "gpt-3.5-turbo"
#
#ULTRA_STRICT_PROMPT = """You are an Information Extraction Engine … <твой prompt>"""
#QA_PROMPT_SYSTEM = "You are a helpful assistant for answering questions based on given context."
#SUMMARIZE_PROMPT_SYSTEM = "You are a summarizer. Summarize the user's input concisely."

import requests

async def gptunnel_chat_completion(
    messages: List[Dict],
    model: str = GPTUNNEL_MODEL,
    max_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    **kwargs
) -> Optional[str]:
    allowed = {k: v for k, v in kwargs.items() if k in ("stop", "n", "model", "max_tokens", "temperature", "top_p")}
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        **allowed
    }
    headers = {
        "Authorization": f"Bearer {GPTUNNEL_API_KEY}",
        "Content-Type": "application/json"
    }
    resp = requests.post(GPTUNNEL_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    js = resp.json()
    try:
        return js["choices"][0]["message"]["content"]
    except Exception as e:
        logging.error("GPTunnel response parsing error: %s", e)
        return None

# … ваш импорт и настройки …

async def llm_model_func(
    prompt: str,
    system_prompt: Optional[str] = None,  # <-- добавлено
    history_messages = None,  # можно, если LightRAG передаёт историю
    **kwargs
) -> str:
    # Если LightRAG передал system_prompt, используем его, иначе fallback
    sp = system_prompt or "You are an assistant."

    messages = [
        {"role": "system", "content": sp},
        {"role": "user", "content": prompt}
    ]

    raw = await gptunnel_chat_completion(messages=messages, **kwargs)
    if raw is None:
        return ""
    # Мы не делаем отдельную post‑обработку extract здесь;
    # LightRAG будет уметь мержить сущности, если system_prompt — строгий.
    return raw.strip()


def fix_extraction_output(text: str) -> str:
    text = text.replace("<|COMPLETE|>", "")
    lines = text.splitlines()
    out = []
    entity_pattern = re.compile(r"^entity#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$")
    relation_pattern = re.compile(r"^relation#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$")
    for l in lines:
        l2 = l.strip()
        if entity_pattern.match(l2) or relation_pattern.match(l2):
            out.append(l2)
    out.append("<|COMPLETE|>")
    return "\n".join(out)

def simple_split(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = min(length, start + chunk_size)
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def soft_clean(text: str) -> str:
    return text.strip().replace("\n", " ").replace("\r", " ")

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

async def embedding_func(texts):
    return embed_model.encode(texts, convert_to_numpy=True)

def init_lightrag(work_dir: str = "data/rag_data") -> LightRAG:
    rag = LightRAG(
        working_dir=work_dir,
        llm_model_func=llm_model_func,
        embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, func=embedding_func),
        vector_storage="FaissVectorDBStorage",
    )
    return rag

async def load_dataset_into_lightrag(rag: LightRAG, df: pd.DataFrame, text_column: str = "text_clean"):
    batch = []
    for _, row in df.iterrows():
        text = row[text_column]
        if not isinstance(text, str):
            text = str(text)
        cleaned = soft_clean(text)
        chunks = simple_split(cleaned)
        batch.extend(chunks)
        if len(batch) >= 100:
            await rag.ainsert(batch)
            batch = []
    if batch:
        await rag.ainsert(batch)

async def main():
    #df = pd.DataFrame({"text_clean": [
    #    "Apple Inc. is a company based in Cupertino. Tim Cook is the CEO.",
    #    "Gucci is a fashion brand. Its logo features a stylized G."
    #]})

    rag = init_lightrag()
    await rag.initialize_storages()
    await initialize_pipeline_status()
    await load_dataset_into_lightrag(rag, df)

#if __name__ == "__main__":
#    asyncio.run(main())
#wait main()

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
    #df = pd.DataFrame({"text_clean": [
    #    "Apple Inc. is a company based in Cupertino. Tim Cook is the CEO.",
    #    "Gucci is a fashion brand. Its logo features a stylized G."
    #]})
    df = pd.read_csv('data/channel_data.csv')

    rag = init_lightrag()
    await rag.initialize_storages()
    await initialize_pipeline_status()


INFO: [_] Loaded graph from data/rag_data/graph_chunk_entity_relation.graphml with 5 nodes, 0 edges
INFO: [_] Faiss index loaded with 5 vectors from data/rag_data/faiss_index_entities.index
INFO: [_] Faiss index loaded with 0 vectors from data/rag_data/faiss_index_relationships.index
INFO: [_] Faiss index loaded with 2 vectors from data/rag_data/faiss_index_chunks.index
INFO: [_] Process 51167 KV load full_docs with 2 records
INFO: [_] Process 51167 KV load text_chunks with 2 records
INFO: [_] Process 51167 KV load full_entities with 2 records
INFO: [_] Process 51167 KV load full_relations with 0 records
INFO: [_] Process 51167 KV load entity_chunks with 5 records
INFO: [_] Process 51167 KV load relation_chunks with 0 records
INFO: [_] Process 51167 KV load llm_response_cache with 5 records
INFO: [_] Process 51167 doc status load doc_status with 2 records


In [ ]:
    await load_dataset_into_lightrag(rag, df, text_column = "text_clean")

INFO: Reset 26 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 233 document(s)
INFO: Extracting stage 1/233: unknown_source
INFO: Processing d-id: doc-8de5964ebbafd676130c423bf0a93d5f
INFO: Extracting stage 2/233: unknown_source
INFO: Processing d-id: doc-47e2e8c3fdb7739e740b95345a803cac
INFO:  == LLM cache == saving: default:extract:ef796c4a6ffd03227f9ddf32f1c12a02
INFO:  == LLM cache == saving: default:extract:603dc3fcce17042e57230eaf145b811c
INFO:  == LLM cache == saving: default:extract:b2af869ed464f54dfea81b651c44c790
INFO: Chunk 1 of 1 extracted 3 Ent + 0 Rel chunk-8de5964ebbafd676130c423bf0a93d5f
INFO:  == LLM cache == saving: default:extract:4f61f996c0735a6bf9ded3e8c8673cf3
INFO: Chunk 1 of 1 extracted 1 Ent + 0 Rel chunk-47e2e8c3fdb7739e740b95345a803cac
INFO: Merging stage 1/233: unknown_source
INFO: Phase 1: Processing 3 entities from doc-8de5964ebbafd676130c423bf0a93d5f (async: 8)
INFO: Merging stage 2/233: unknown_source
INFO: Phase 1: Processing 1 entit

CancelledError: 

INFO:  == LLM cache == saving: default:extract:1c6a3abeb7c1d0ff2aedd773d8d65583


KeyError: 'text_clean'

In [10]:
param = QueryParam(
    mode="hybrid",
    user_prompt=ULTRA_STRICT_PROMPT
)
extraction = await rag.aquery("Как стать разработчиком Flutter?", param=param)
print(extraction)

INFO:  == LLM cache == saving: hybrid:keywords:74cda26e095cb5a797680f617b040899
INFO: Query nodes: Programming skills, Dart language proficiency, App development, Flutter framework (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 17 relations
INFO: Query edges: Flutter developer, Career path in Flutter development, Becoming a developer (top_k:40, cosine:0.2)
INFO: Global query: 50 entites, 40 relations
INFO: Raw search results: 86 entities, 49 relations, 0 vector chunks
INFO: After truncation: 60 entities, 49 relations
INFO: Selecting 150 from 634 entity-related chunks by vector similarity
INFO: Find 33 additional chunks in 31 relations (deduplicated 11)
INFO: Selecting 33 from 33 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 183 -> 183 (deduplicated 0)
INFO: Final context: 60 entities, 49 relations, 20 chunks
INFO: Final chunks S+F/O: E2/1 R1/1 E1/2 R1/2 E1/3 R2/3 E1/4 R1/4 E1/5 R1/5 E2/6 R1/6 E2/7 R1/7 E1/8 R1/8 E1/9 R1/9 E1/10 R1/10
INFO:  == LLM

### Answer:

Для становления разработчиком Flutter необходимо о behalten bestimmte Schritte und Qualifikationen. Hier sind einige Schritte, die Sie befolgen können, um ein Flutter-Entwickler zu werden:

1. **Erlernen Sie Dart**: Dart ist die Programmiersprache, die mit Flutter verwendet wird. Es ist wichtig, ein solides Verständnis von Dart zu haben, um effektiv mit Flutter zu arbeiten.

2. **Studieren Sie Flutter**: Vertiefen Sie Ihr Wissen über Flutter, indem Sie Tutorials, Kurse und Dokumentationen durchgehen. Praktische Erfahrung ist entscheidend, um Ihre Fähigkeiten zu verbessern.

3. **Entwickeln Sie Projekte**: Starten Sie eigene Projekte, um Ihre Fähigkeiten zu üben und Ihr Portfolio aufzubauen. Dies hilft Ihnen auch, praktische Erfahrung zu sammeln.

4. **Community-Engagement**: Treten Sie der Flutter-Community bei, besuchen Sie Meetups, Foren und Konferenzen, um sich mit anderen Entwicklern auszutauschen und von deren Erfahrungen
